In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Feature engineering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import (VarianceThreshold, mutual_info_regression, 
                                       RFE, SelectKBest, f_regression)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor

# For saving features
import json
import pickle

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Libraries imported successfully")


✅ Libraries imported successfully


In [2]:
# Load datasets
relationship_path = '../data/relationship/'
master_df = pd.read_csv(os.path.join(relationship_path, 'master_relationship_table.csv'))

print("="*60)
print("MASTER DATASET LOADED")
print("="*60)
print(f"Shape: {master_df.shape}")
print(f"Years: {master_df['Year'].unique().tolist()}")
print(f"Categories: {master_df['Category'].nunique()} categories")
print(f"Columns: {master_df.columns.tolist()}")
print("\nFirst 5 rows:")
print(master_df.head())

MASTER DATASET LOADED
Shape: (288, 15)
Years: [2020, 2021, 2022, 2023, 2024, 2025]
Categories: 48 categories
Columns: ['Year', 'Category', 'Spending_Lowest_20', 'Spending_Second_20', 'Spending_Middle_20', 'Spending_Fourth_20', 'Spending_Highest_20', 'Crude_Oil_Price_USD', 'Petrol_Price_pence_per_litre', 'Diesel_Price_pence_per_litre', 'CPI_Fuels_and_Lubricants', 'CPI_Electricity', 'CPI_Gas', 'CPI_Liquid_Fuels', 'CPI_Solid_Fuels']

First 5 rows:
   Year          Category  Spending_Lowest_20  Spending_Second_20  \
0  2020  Alcoholic drinks                4.10                5.44   
1  2021  Alcoholic drinks                4.65                6.50   
2  2022  Alcoholic drinks                4.61                6.81   
3  2023  Alcoholic drinks                4.74                6.78   
4  2024  Alcoholic drinks                4.94                7.16   

   Spending_Middle_20  Spending_Fourth_20  Spending_Highest_20  \
0                7.63                9.73                14.92   
1   

In [3]:
# HANDLE MISSING VALUES
def prepare_master_data(df):
   
    df_clean = df.copy()
    
    # 1. Handle CPI columns - fill with reasonable baseline values
    cpi_cols = ['CPI_Fuels_and_Lubricants', 'CPI_Electricity', 'CPI_Gas', 
                'CPI_Liquid_Fuels', 'CPI_Solid_Fuels']
    
    # Set baseline CPI values (2015=100) with realistic estimates
    # These are approximate values based on UK CPIH trends
    cpi_estimates = {
        2020: {'CPI_Fuels_and_Lubricants': 95.0, 'CPI_Electricity': 98.0, 
               'CPI_Gas': 97.0, 'CPI_Liquid_Fuels': 92.0, 'CPI_Solid_Fuels': 96.0},
        2021: {'CPI_Fuels_and_Lubricants': 102.0, 'CPI_Electricity': 105.0, 
               'CPI_Gas': 104.0, 'CPI_Liquid_Fuels': 100.0, 'CPI_Solid_Fuels': 102.0},
        2022: {'CPI_Fuels_and_Lubricants': 120.0, 'CPI_Electricity': 130.0, 
               'CPI_Gas': 135.0, 'CPI_Liquid_Fuels': 125.0, 'CPI_Solid_Fuels': 115.0},
        2023: {'CPI_Fuels_and_Lubricants': 115.0, 'CPI_Electricity': 125.0, 
               'CPI_Gas': 130.0, 'CPI_Liquid_Fuels': 118.0, 'CPI_Solid_Fuels': 110.0},
        2024: {'CPI_Fuels_and_Lubricants': 112.0, 'CPI_Electricity': 118.0, 
               'CPI_Gas': 122.0, 'CPI_Liquid_Fuels': 115.0, 'CPI_Solid_Fuels': 108.0},
        2025: {'CPI_Fuels_and_Lubricants': 110.0, 'CPI_Electricity': 115.0, 
               'CPI_Gas': 118.0, 'CPI_Liquid_Fuels': 112.0, 'CPI_Solid_Fuels': 106.0}
    }
    
    # Apply CPI estimates by year
    for year, values in cpi_estimates.items():
        for col, value in values.items():
            df_clean.loc[df_clean['Year'] == year, col] = value
    
    # For any remaining NaN values, use interpolation
    for col in cpi_cols:
        df_clean[col] = df_clean.groupby('Category')[col].transform(
            lambda x: x.interpolate(method='linear', limit_direction='both')
        )
        df_clean[col] = df_clean[col].fillna(df_clean[col].mean())
    
    # 2. Handle Oil Price missing (2020)
    # Use 2021 oil price as proxy for 2020
    oil_2021 = df_clean[df_clean['Year'] == 2021]['Crude_Oil_Price_USD'].iloc[0] 
    df_clean.loc[df_clean['Year'] == 2020, 'Crude_Oil_Price_USD'] = oil_2021
    
    # If still any missing, fill with overall mean
    df_clean['Crude_Oil_Price_USD'] = df_clean['Crude_Oil_Price_USD'].fillna(
        df_clean['Crude_Oil_Price_USD'].mean()
    )
    
    # 3. Verify no missing values remain
    print("\n🔹 Missing Values Check:")
    missing = df_clean.isnull().sum()
    missing = missing[missing > 0]
    
    if len(missing) == 0:
        print("✅ All missing values handled successfully!")
    else:
        print("⚠️ Still missing values:")
        print(missing)
    
    return df_clean

# Apply the quick fix
master_df_clean = prepare_master_data(master_df)

print(f"\n✅ Master data prepared. Shape: {master_df_clean.shape}")

# Show sample with CPI values
print("\n📊 Sample of data with CPI values:")
sample_cols = ['Year', 'Category', 'CPI_Fuels_and_Lubricants', 
               'CPI_Electricity', 'CPI_Gas', 'Crude_Oil_Price_USD']
print(master_df_clean[sample_cols].head(10))


🔹 Missing Values Check:
✅ All missing values handled successfully!

✅ Master data prepared. Shape: (288, 15)

📊 Sample of data with CPI values:
   Year                Category  CPI_Fuels_and_Lubricants  CPI_Electricity  \
0  2020        Alcoholic drinks                      95.0             98.0   
1  2021        Alcoholic drinks                     102.0            105.0   
2  2022        Alcoholic drinks                     120.0            130.0   
3  2023        Alcoholic drinks                     115.0            125.0   
4  2024        Alcoholic drinks                     112.0            118.0   
5  2025        Alcoholic drinks                     110.0            115.0   
6  2020  Audio-visual equipment                      95.0             98.0   
7  2021  Audio-visual equipment                     102.0            105.0   
8  2022  Audio-visual equipment                     120.0            130.0   
9  2023  Audio-visual equipment                     115.0            125.0 

In [ ]:
# PART 3: FEATURE ENGINEERING
def create_all_features(df):
    df_feat = df.copy()
    # TEMPORAL FEATURES
    # Year is already in the data
    df_feat['Year'] = df_feat['Year']
    
    # Create seasonality flags
    df_feat['Is_2020'] = (df_feat['Year'] == 2020).astype(int)
    df_feat['Is_2021'] = (df_feat['Year'] == 2021).astype(int)
    df_feat['Is_2022'] = (df_feat['Year'] == 2022).astype(int)
    df_feat['Is_2023'] = (df_feat['Year'] == 2023).astype(int)
    df_feat['Is_2024'] = (df_feat['Year'] == 2024).astype(int)
    df_feat['Is_2025'] = (df_feat['Year'] == 2025).astype(int)
    
    # Event flags
    df_feat['Is_COVID_Year'] = (df_feat['Year'].isin([2020, 2021])).astype(int)
    df_feat['Is_Post_COVID'] = (df_feat['Year'] >= 2022).astype(int)
    df_feat['Is_Ukraine_War_Year'] = (df_feat['Year'] >= 2022).astype(int)
    df_feat['Is_Pre_War'] = (df_feat['Year'] < 2022).astype(int)
    
    # Time since events (in years)
    df_feat['Years_Since_2020'] = df_feat['Year'] - 2020
    df_feat['Years_Since_2022'] = np.maximum(df_feat['Year'] - 2022, 0)
    
    
    # NUMERICAL FEATURES - TRANSFORMATIONS
    # Log transformations (handle zeros by adding small epsilon)
    eps = 1e-6
    
    # Spending transformations
    spending_cols = ['Spending_Lowest_20', 'Spending_Second_20', 'Spending_Middle_20', 
                     'Spending_Fourth_20', 'Spending_Highest_20']
    
    for col in spending_cols:
        if col in df_feat.columns:
            df_feat[f'{col}_Log'] = np.log(df_feat[col] + eps)
            df_feat[f'{col}_Sqrt'] = np.sqrt(df_feat[col])
    
    # Price transformations
    price_cols = ['Crude_Oil_Price_USD', 'Petrol_Price_pence_per_litre', 
                  'Diesel_Price_pence_per_litre']
    
    for col in price_cols:
        if col in df_feat.columns:
            df_feat[f'{col}_Log'] = np.log(df_feat[col] + eps)
            df_feat[f'{col}_Sqrt'] = np.sqrt(df_feat[col])
    
    # CPI transformations
    cpi_cols = ['CPI_Fuels_and_Lubricants', 'CPI_Electricity', 'CPI_Gas', 
                'CPI_Liquid_Fuels', 'CPI_Solid_Fuels']
    
    for col in cpi_cols:
        if col in df_feat.columns:
            df_feat[f'{col}_Log'] = np.log(df_feat[col] + eps)
    
    
    # RATIO FEATURES (CRITICAL FOR COMPARATIVE ANALYSIS)
    
    
    # Spending ratios (Lowest vs Middle - YOUR KEY METRIC)
    df_feat['Spending_Lowest_to_Middle_Ratio'] = (
        df_feat['Spending_Lowest_20'] / (df_feat['Spending_Middle_20'] + eps)
    )
    
    df_feat['Spending_Lowest_to_Highest_Ratio'] = (
        df_feat['Spending_Lowest_20'] / (df_feat['Spending_Highest_20'] + eps)
    )
    
    df_feat['Spending_Middle_to_Highest_Ratio'] = (
        df_feat['Spending_Middle_20'] / (df_feat['Spending_Highest_20'] + eps)
    )
    
    # Spending gaps (absolute differences)
    df_feat['Spending_Gap_Lowest_Middle'] = (
        df_feat['Spending_Middle_20'] - df_feat['Spending_Lowest_20']
    )
    
    df_feat['Spending_Gap_Lowest_Highest'] = (
        df_feat['Spending_Highest_20'] - df_feat['Spending_Lowest_20']
    )
    
    df_feat['Spending_Gap_Middle_Highest'] = (
        df_feat['Spending_Highest_20'] - df_feat['Spending_Middle_20']
    )
    
    # Price ratios (pass-through analysis)
    df_feat['Oil_to_Petrol_Ratio'] = (
        df_feat['Crude_Oil_Price_USD'] / (df_feat['Petrol_Price_pence_per_litre'] + eps)
    )
    
    df_feat['Petrol_to_Diesel_Ratio'] = (
        df_feat['Petrol_Price_pence_per_litre'] / (df_feat['Diesel_Price_pence_per_litre'] + eps)
    )
    
    df_feat['Oil_to_CPI_Ratio'] = (
        df_feat['Crude_Oil_Price_USD'] / (df_feat['CPI_Fuels_and_Lubricants'] + eps)
    )
    
    
    # CATEGORY ENCODING
    
    
    # Frequency encoding
    category_counts = df_feat['Category'].value_counts()
    df_feat['Category_Frequency'] = df_feat['Category'].map(category_counts)
    
    # Label encoding
    le = LabelEncoder()
    df_feat['Category_Encoded'] = le.fit_transform(df_feat['Category'])
    
    # One-hot encoding (for top categories only to avoid explosion)
    top_categories = df_feat['Category'].value_counts().head(10).index
    for cat in top_categories:
        df_feat[f'Category_{cat.replace(" ", "_")}'] = (df_feat['Category'] == cat).astype(int)
    
    # Category type flags (essential vs discretionary)
    essential_categories = ['Electricity', 'Gas', 'Food', 'Bread, rice and cereals', 
                           'Milk and dairy', 'Vegetables', 'Fruit', 'Petrol, diesel and oils']
    
    df_feat['Is_Essential_Category'] = df_feat['Category'].isin(essential_categories).astype(int)
    
    energy_categories = ['Electricity', 'Gas', 'Other fuels', 'Petrol, diesel and oils']
    df_feat['Is_Energy_Category'] = df_feat['Category'].isin(energy_categories).astype(int)
    
    
    # DOMAIN-SPECIFIC FEATURES
    
    
    # Energy intensity score (how exposed is this category to oil prices)
    energy_exposure = {
        'Petrol, diesel and oils': 1.0,
        'Electricity': 0.8,
        'Gas': 0.7,
        'Other fuels': 0.9,
        'Public transport': 0.6,
        'Vehicle purchase': 0.5,
        'Vehicle repairs': 0.3,
        'Food': 0.4,
        'Restaurant meals': 0.2,
        'Take-away food': 0.2,
        'Package holidays': 0.3,
        'Clothing': 0.1,
        'Footwear': 0.1,
        'Furniture and furnishings': 0.1,
        'Household appliances': 0.1,
        'Medical products': 0.1,
        'Personal care': 0.05,
        'Insurance': 0.05,
        'Education': 0.0,
        'Health': 0.0,
        'Communication': 0.0,
        'Recreation': 0.0
    }
    
    df_feat['Energy_Exposure_Score'] = df_feat['Category'].map(
        lambda x: energy_exposure.get(x, 0.1)
    )
    
    # Price pass-through estimate (simplified)
    df_feat['Estimated_Pass_Through'] = (
        df_feat['Energy_Exposure_Score'] * df_feat['Crude_Oil_Price_USD'].pct_change()
    )
    
    # Fuel poverty indicator (based on spending > threshold)
    fuel_poverty_threshold = df_feat['Spending_Lowest_20'].quantile(0.75)
    df_feat['Fuel_Poverty_Risk_Lowest'] = (
        df_feat['Spending_Lowest_20'] > fuel_poverty_threshold
    ).astype(int)
    
    # Displacement ratio (energy vs food spending)
    # We'll create this per category by comparing to food
    food_spending = df_feat[df_feat['Category'] == 'Food'][['Year', 'Spending_Lowest_20']].rename(
        columns={'Spending_Lowest_20': 'Food_Spending_Lowest'}
    )
    df_feat = df_feat.merge(food_spending, on='Year', how='left')
    df_feat['Energy_to_Food_Ratio_Lowest'] = (
        df_feat['Spending_Lowest_20'] / (df_feat['Food_Spending_Lowest'] + eps)
    )
    df_feat = df_feat.drop('Food_Spending_Lowest', axis=1)
    
    
    # INTERACTION FEATURES (YOUR KEY DIFFERENTIATOR)
    
    
    # Oil price interactions with income groups
    df_feat['Oil_Price_x_Lowest_20'] = (
        df_feat['Crude_Oil_Price_USD'] * df_feat['Spending_Lowest_20']
    )
    
    df_feat['Oil_Price_x_Middle_20'] = (
        df_feat['Crude_Oil_Price_USD'] * df_feat['Spending_Middle_20']
    )
    
    df_feat['Oil_Price_x_Highest_20'] = (
        df_feat['Crude_Oil_Price_USD'] * df_feat['Spending_Highest_20']
    )
    
    # Petrol price interactions
    df_feat['Petrol_Price_x_Lowest_20'] = (
        df_feat['Petrol_Price_pence_per_litre'] * df_feat['Spending_Lowest_20']
    )
    
    df_feat['Petrol_Price_x_Middle_20'] = (
        df_feat['Petrol_Price_pence_per_litre'] * df_feat['Spending_Middle_20']
    )
    
    # Category × Year interactions
    df_feat['Category_Year_Interaction'] = (
        df_feat['Category_Encoded'] * (df_feat['Year'] - 2020)
    )
    
    # Energy category × Oil price
    df_feat['Energy_Category_x_Oil'] = (
        df_feat['Is_Energy_Category'] * df_feat['Crude_Oil_Price_USD']
    )
    
    # Essential category × Spending gap
    df_feat['Essential_x_Spending_Gap'] = (
        df_feat['Is_Essential_Category'] * df_feat['Spending_Gap_Lowest_Highest']
    )
    
    
    # LAGGED FEATURES (Time series)
    
    
    # Sort by Year and Category to create lags correctly
    df_feat = df_feat.sort_values(['Category', 'Year'])
    
    lag_cols = ['Crude_Oil_Price_USD', 'Petrol_Price_pence_per_litre', 
                'Diesel_Price_pence_per_litre', 'CPI_Fuels_and_Lubricants']
    
    for col in lag_cols:
        if col in df_feat.columns:
            # Lag within each category
            df_feat[f'{col}_Lag1'] = df_feat.groupby('Category')[col].shift(1)
            df_feat[f'{col}_Lag2'] = df_feat.groupby('Category')[col].shift(2)
            df_feat[f'{col}_Lag3'] = df_feat.groupby('Category')[col].shift(3)
    
    
    # ROLLING WINDOW FEATURES
    
    
    # Rolling means and std (3-year window)
    for col in ['Spending_Lowest_20', 'Spending_Middle_20', 'Spending_Highest_20']:
        if col in df_feat.columns:
            df_feat[f'{col}_Rolling_Mean_3yr'] = (
                df_feat.groupby('Category')[col].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
            )
            df_feat[f'{col}_Rolling_Std_3yr'] = (
                df_feat.groupby('Category')[col].rolling(3, min_periods=1).std().reset_index(level=0, drop=True)
            )
    
    
    # CHANGE/DELTA FEATURES (For causal analysis)
    
    
    # Year-over-year changes for spending
    for col in ['Spending_Lowest_20', 'Spending_Middle_20', 'Spending_Highest_20']:
        if col in df_feat.columns:
            df_feat[f'{col}_YoY_Change'] = (
                df_feat.groupby('Category')[col].pct_change()
            )
            df_feat[f'{col}_YoY_Abs_Change'] = (
                df_feat.groupby('Category')[col].diff()
            )
    
    # Year-over-year changes for oil price
    if 'Crude_Oil_Price_USD' in df_feat.columns:
        df_feat['Oil_Price_YoY_Change'] = (
            df_feat.groupby('Category')['Crude_Oil_Price_USD'].pct_change()
        )
    
    
    # TARGET VARIABLES (For all 3 approaches)
    
    
    # Target 1: Spending Gap (Primary - Regression)
    df_feat['Target_Spending_Gap'] = (
        df_feat['Spending_Highest_20'] - df_feat['Spending_Lowest_20']
    )
    
    # Target 2: Low income energy spending (Regression)
    df_feat['Target_Lowest_Energy_Spending'] = df_feat['Spending_Lowest_20']
    
    # Target 3: Middle income energy spending (Regression)
    df_feat['Target_Middle_Energy_Spending'] = df_feat['Spending_Middle_20']
    
    # Target 4: YoY change in gap (Causal)
    df_feat['Target_Gap_YoY_Change'] = (
        df_feat.groupby('Category')['Target_Spending_Gap'].pct_change()
    )
    
    # Target 5: Energy Poverty Flag (Classification)
    threshold = df_feat['Spending_Lowest_20'].quantile(0.75)
    df_feat['Target_Energy_Poverty'] = (
        df_feat['Spending_Lowest_20'] > threshold
    ).astype(int)
    
    # Target 6: Crisis Flag (Classification)
    if 'Spending_Lowest_20_YoY_Change' in df_feat.columns:
        df_feat['Target_Crisis_Flag'] = (
            df_feat['Spending_Lowest_20_YoY_Change'] > 0.20
        ).astype(int)
    
    return df_feat

# Apply feature engineering
master_featured = create_all_features(master_df)
print(f"\n✅ Feature engineering complete!")
print(f"New shape: {master_featured.shape}")
print(f"Total features: {len(master_featured.columns)}")
print(f"\nColumns added: {len(master_featured.columns) - len(master_df.columns)} new features")


✅ Feature engineering complete!
New shape: (288, 114)
Total features: 114

Columns added: 99 new features


In [5]:
print(master_featured.columns.to_list())

['Year', 'Category', 'Spending_Lowest_20', 'Spending_Second_20', 'Spending_Middle_20', 'Spending_Fourth_20', 'Spending_Highest_20', 'Crude_Oil_Price_USD', 'Petrol_Price_pence_per_litre', 'Diesel_Price_pence_per_litre', 'CPI_Fuels_and_Lubricants', 'CPI_Electricity', 'CPI_Gas', 'CPI_Liquid_Fuels', 'CPI_Solid_Fuels', 'Is_2020', 'Is_2021', 'Is_2022', 'Is_2023', 'Is_2024', 'Is_2025', 'Is_COVID_Year', 'Is_Post_COVID', 'Is_Ukraine_War_Year', 'Is_Pre_War', 'Years_Since_2020', 'Years_Since_2022', 'Spending_Lowest_20_Log', 'Spending_Lowest_20_Sqrt', 'Spending_Second_20_Log', 'Spending_Second_20_Sqrt', 'Spending_Middle_20_Log', 'Spending_Middle_20_Sqrt', 'Spending_Fourth_20_Log', 'Spending_Fourth_20_Sqrt', 'Spending_Highest_20_Log', 'Spending_Highest_20_Sqrt', 'Crude_Oil_Price_USD_Log', 'Crude_Oil_Price_USD_Sqrt', 'Petrol_Price_pence_per_litre_Log', 'Petrol_Price_pence_per_litre_Sqrt', 'Diesel_Price_pence_per_litre_Log', 'Diesel_Price_pence_per_litre_Sqrt', 'CPI_Fuels_and_Lubricants_Log', 'CPI_El

In [ ]:
# FEATURE SELECTION WITH  NaN HANDLING

def prepare_for_feature_selection_fixed(df):
   
    
    # Identify feature columns (exclude non-features)
    exclude_cols = ['Year', 'Category', 'Spending_Lowest_20', 'Spending_Second_20',
                    'Spending_Middle_20', 'Spending_Fourth_20', 'Spending_Highest_20',
                    'Target_Spending_Gap', 'Target_Lowest_Energy_Spending',
                    'Target_Middle_Energy_Spending', 'Target_Gap_YoY_Change',
                    'Target_Energy_Poverty', 'Target_Crisis_Flag']
    
    # Get all numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Feature columns (exclude targets and identifiers)
    feature_cols = [col for col in numeric_cols if col not in exclude_cols]
    
    print(f"\n🔹 Feature columns: {len(feature_cols)}")
    print(f"Sample features: {feature_cols[:10]}")
    
    # Create feature matrix
    X = df[feature_cols].copy()
    
    # Check for NaN values before handling
    print(f"\n🔹 NaN values before handling: {X.isnull().sum().sum()}")
    
    # HANDLE NaN VALUES ROBUSTLY
    # 1. First, fill with column means
    for col in X.columns:
        if X[col].isnull().any():
            X[col] = X[col].fillna(X[col].mean())
    
    # 2. Handle inf values
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        if X[col].isnull().any():
            X[col] = X[col].fillna(X[col].mean())
    
    # 3. Final check - if any NaN remain, fill with 0
    X = X.fillna(0)
    
    # Verify no NaN remain
    print(f"🔹 NaN values after handling: {X.isnull().sum().sum()}")
    print(f"✅ All NaN values handled")
    
    # Target (using Spending Gap as primary target)
    y = df['Target_Spending_Gap'].copy()
    
    # Handle NaN in target
    if y.isnull().any():
        print(f"⚠️ Target has {y.isnull().sum()} NaN values - filling with mean")
        y = y.fillna(y.mean())
    
    return X, y, feature_cols

# Run the fixed preparation
X_features, y_target, feature_names = prepare_for_feature_selection_fixed(master_featured)
print(f"\n✅ Feature matrix prepared: {X_features.shape}")
print(f"Target shape: {y_target.shape}")
print(f"X_features columns: {X_features.columns.tolist()[:10]}...")

# Now run the correlation analysis
def correlation_analysis_fixed(X, y, top_n=20):
    """Analyze correlation with target with error handling"""
    
    correlations = []
    for col in X.columns:
        try:
            corr_val = X[col].corr(y)
            if not np.isnan(corr_val):
                correlations.append({'Feature': col, 'Correlation': corr_val})
        except:
            pass
    
    correlations_df = pd.DataFrame(correlations)
    if len(correlations_df) > 0:
        correlations_df['Abs_Correlation'] = np.abs(correlations_df['Correlation'])
        correlations_df = correlations_df.sort_values('Abs_Correlation', ascending=False)
        
        print("\n" + "="*60)
        print("TOP 20 FEATURES BY CORRELATION WITH TARGET")
        print("="*60)
        print(correlations_df.head(top_n).to_string())
    else:
        print("⚠️ No valid correlations found")
        correlations_df = pd.DataFrame(columns=['Feature', 'Correlation', 'Abs_Correlation'])
    
    return correlations_df

corr_results = correlation_analysis_fixed(X_features, y_target)

# FIXED MUTUAL INFORMATION
def mutual_information_analysis_fixed(X, y, top_n=20):
    
    
    try:
        # Ensure no NaN values (should already be handled but double check)
        X_clean = X.fillna(0)
        y_clean = y.fillna(y.mean())
        
        mi_scores = mutual_info_regression(X_clean, y_clean, random_state=42)
        
        mi_df = pd.DataFrame({
            'Feature': X.columns,
            'MI_Score': mi_scores
        }).sort_values('MI_Score', ascending=False)
        
        print("\n" + "="*60)
        print("TOP 20 FEATURES BY MUTUAL INFORMATION")
        print("="*60)
        print(mi_df.head(top_n).to_string())
        
        return mi_df
    except Exception as e:
        print(f"⚠️ Mutual information calculation failed: {e}")
        # Return empty dataframe with features
        return pd.DataFrame({'Feature': X.columns, 'MI_Score': 0})

mi_results = mutual_information_analysis_fixed(X_features, y_target)

# FIXED RANDOM FOREST
def random_forest_importance_fixed(X, y, top_n=20):
    
    
    try:
        # Use subset for performance
        n_features = min(X.shape[1], 100)
        X_subset = X.iloc[:, :n_features]
        
        rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf.fit(X_subset, y)
        
        importance_df = pd.DataFrame({
            'Feature': X_subset.columns,
            'Importance': rf.feature_importances_
        }).sort_values('Importance', ascending=False)
        
        print("\n" + "="*60)
        print("TOP 20 FEATURES BY RANDOM FOREST IMPORTANCE")
        print("="*60)
        print(importance_df.head(top_n).to_string())
        
        return importance_df, rf
    except Exception as e:
        print(f"⚠️ Random Forest calculation failed: {e}")
        return pd.DataFrame({'Feature': X.columns[:min(X.shape[1], 100)], 'Importance': 0}), None

rf_importance, rf_model = random_forest_importance_fixed(X_features, y_target)

# FIXED LASSO
def lasso_selection_fixed(X, y, top_n=20):
    
    
    try:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        lasso = LassoCV(cv=5, random_state=42, n_jobs=-1)
        lasso.fit(X_scaled, y)
        
        lasso_df = pd.DataFrame({
            'Feature': X.columns,
            'Coefficient': lasso.coef_
        })
        lasso_df['Abs_Coefficient'] = np.abs(lasso_df['Coefficient'])
        lasso_df = lasso_df.sort_values('Abs_Coefficient', ascending=False)
        lasso_df = lasso_df[lasso_df['Coefficient'] != 0]
        
        print("\n" + "="*60)
        print("TOP 20 FEATURES BY LASSO COEFFICIENT")
        print("="*60)
        print(lasso_df.head(top_n).to_string())
        
        return lasso_df, lasso
    except Exception as e:
        print(f"⚠️ Lasso calculation failed: {e}")
        return pd.DataFrame({'Feature': X.columns, 'Coefficient': 0, 'Abs_Coefficient': 0}), None

lasso_results, lasso_model = lasso_selection_fixed(X_features, y_target)

# FIXED COMPILE SUMMARY
def compile_feature_selection_summary_fixed(corr, mi, rf, lasso):
    
    
    # Create base dataframe from correlation results
    if len(corr) > 0:
        summary = pd.DataFrame({'Feature': corr['Feature'].tolist()})
    else:
        summary = pd.DataFrame({'Feature': X_features.columns.tolist()})
    
    # Merge all results
    summary = summary.merge(
        corr[['Feature', 'Abs_Correlation']].rename(columns={'Abs_Correlation': 'Correlation_Score'}),
        on='Feature', how='left'
    )
    
    summary = summary.merge(
        mi[['Feature', 'MI_Score']],
        on='Feature', how='left'
    )
    
    summary = summary.merge(
        rf[['Feature', 'Importance']],
        on='Feature', how='left'
    )
    
    summary = summary.merge(
        lasso[['Feature', 'Abs_Coefficient']].rename(columns={'Abs_Coefficient': 'Lasso_Score'}),
        on='Feature', how='left'
    )
    
    # Fill NaN values with 0
    summary = summary.fillna(0)
    
    # Calculate composite score
    for col in ['Correlation_Score', 'MI_Score', 'Importance', 'Lasso_Score']:
        if col in summary.columns:
            max_val = summary[col].max()
            min_val = summary[col].min()
            if max_val > min_val:
                summary[f'{col}_Normalized'] = (summary[col] - min_val) / (max_val - min_val)
            else:
                summary[f'{col}_Normalized'] = 0
    
    score_cols = [col for col in summary.columns if 'Normalized' in col]
    if score_cols:
        summary['Composite_Score'] = summary[score_cols].mean(axis=1)
        summary = summary.sort_values('Composite_Score', ascending=False)
    
    print("\n" + "="*60)
    print("FEATURE SELECTION SUMMARY - TOP 20 FEATURES")
    print("="*60)
    print(summary.head(20).to_string())
    
    return summary

selection_summary = compile_feature_selection_summary_fixed(
    corr_results, mi_results, rf_importance, lasso_results
)

print(f"\n✅ Feature selection complete!")
print(f"Top 5 features identified:")
if 'Composite_Score' in selection_summary.columns:
    print(selection_summary[['Feature', 'Composite_Score']].head(5))
else:
    print(selection_summary[['Feature']].head(5))


🔹 Feature columns: 101
Sample features: ['Crude_Oil_Price_USD', 'Petrol_Price_pence_per_litre', 'Diesel_Price_pence_per_litre', 'CPI_Fuels_and_Lubricants', 'CPI_Electricity', 'CPI_Gas', 'CPI_Liquid_Fuels', 'CPI_Solid_Fuels', 'Is_2020', 'Is_2021']

🔹 NaN values before handling: 5954
🔹 NaN values after handling: 0
✅ All NaN values handled

✅ Feature matrix prepared: (288, 101)
Target shape: (288,)
X_features columns: ['Crude_Oil_Price_USD', 'Petrol_Price_pence_per_litre', 'Diesel_Price_pence_per_litre', 'CPI_Fuels_and_Lubricants', 'CPI_Electricity', 'CPI_Gas', 'CPI_Liquid_Fuels', 'CPI_Solid_Fuels', 'Is_2020', 'Is_2021']...

TOP 20 FEATURES BY CORRELATION WITH TARGET
                                 Feature  Correlation  Abs_Correlation
35           Spending_Gap_Lowest_Highest     1.000000         1.000000
36           Spending_Gap_Middle_Highest     0.841023         0.841023
24              Spending_Highest_20_Sqrt     0.834928         0.834928
23               Spending_Highest_20_Log  

In [9]:

# SAVE FEATURES IN BINARY, CSV, JSON FORMATS


def save_features_multiple_formats(df, feature_columns, output_dir='../data/features/'):
    
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Create features dataframe
    features_df = df[feature_columns].copy()
    
    # Also save with metadata (include target and identifiers)
    full_features_df = df[['Year', 'Category'] + feature_columns].copy()
    
    
    # SAVE AS CSV
    
    
    csv_path = os.path.join(output_dir, 'features.csv')
    full_features_df.to_csv(csv_path, index=False)
    print(f"✅ CSV saved: {csv_path}")
    
    
    # SAVE AS JSON
    
    
    # Convert to JSON format
    json_path = os.path.join(output_dir, 'features.json')
    
    # Two formats: records (row-based) and split (column-based)
    
    # Format 1: Records (easy to read)
    json_records_path = os.path.join(output_dir, 'features_records.json')
    full_features_df.to_json(json_records_path, orient='records', indent=2)
    print(f"✅ JSON (records) saved: {json_records_path}")
    
    # Format 2: Split (more efficient for large datasets)
    json_split_path = os.path.join(output_dir, 'features_split.json')
    full_features_df.to_json(json_split_path, orient='split', indent=2)
    print(f"✅ JSON (split) saved: {json_split_path}")
    
    
    # SAVE AS BINARY FORMATS
    
    
    # Format 1: Pickle
    pickle_path = os.path.join(output_dir, 'features.pkl')
    with open(pickle_path, 'wb') as f:
        pickle.dump(full_features_df, f)
    print(f"✅ Pickle saved: {pickle_path}")
    
    # Format 2: Parquet (highly efficient)
    parquet_path = os.path.join(output_dir, 'features.parquet')
    full_features_df.to_parquet(parquet_path, index=False)
    print(f"✅ Parquet saved: {parquet_path}")
    
    # Format 3: Feather (fast read/write)
    feather_path = os.path.join(output_dir, 'features.feather')
    full_features_df.to_feather(feather_path)
    print(f"✅ Feather saved: {feather_path}")
    
    # Format 4: HDF5
    hdf_path = os.path.join(output_dir, 'features.h5')
    full_features_df.to_hdf(hdf_path, key='features', mode='w')
    print(f"✅ HDF5 saved: {hdf_path}")
    
    
    # SAVE FEATURE METADATA
    
    
    # Save feature descriptions
    metadata = {
        'dataset_name': 'UK Household Expenditure Features',
        'created_date': pd.Timestamp.now().isoformat(),
        'total_rows': len(full_features_df),
        'total_features': len(feature_columns),
        'years_covered': df['Year'].unique().tolist(),
        'categories_covered': df['Category'].nunique(),
        'feature_list': feature_columns,
        'feature_count_by_type': {
            'temporal': len([f for f in feature_columns if 'Year' in f or 'Is_' in f]),
            'ratio': len([f for f in feature_columns if 'Ratio' in f]),
            'interaction': len([f for f in feature_columns if 'x_' in f or 'Interaction' in f]),
            'lagged': len([f for f in feature_columns if 'Lag' in f]),
            'rolling': len([f for f in feature_columns if 'Rolling' in f]),
            'transform': len([f for f in feature_columns if '_Log' in f or '_Sqrt' in f]),
            'domain': len([f for f in feature_columns if 'Exposure' in f or 'Pass_Through' in f])
        }
    }
    
    # Save metadata as JSON
    metadata_path = os.path.join(output_dir, 'features_metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✅ Metadata saved: {metadata_path}")
    
    return {
        'csv': csv_path,
        'json_records': json_records_path,
        'json_split': json_split_path,
        'pickle': pickle_path,
        'parquet': parquet_path,
        'feather': feather_path,
        'hdf5': hdf_path,
        'metadata': metadata_path
    }


# SELECT FINAL FEATURE SET


# Get top features from selection summary
top_features = selection_summary.head(30)['Feature'].tolist()

print("\n" + "="*60)
print("SAVING FEATURES")
print("="*60)
print(f"Features to save: {len(top_features)}")
print(f"Sample features: {top_features[:10]}")

# Save features
saved_paths = save_features_multiple_formats(
    master_featured, 
    top_features,
    output_dir='../data/features/'
)

print("\n" + "="*60)
print("✅ ALL FEATURES SAVED SUCCESSFULLY!")
print("="*60)
print("\nSaved files:")
for format_name, path in saved_paths.items():
    print(f"  - {format_name}: {path}")


SAVING FEATURES
Features to save: 30
Sample features: ['Spending_Gap_Lowest_Highest', 'Spending_Highest_20_Sqrt', 'Spending_Highest_20_Log', 'Spending_Gap_Middle_Highest', 'Spending_Fourth_20_Log', 'Spending_Highest_20_Rolling_Mean_3yr', 'Spending_Middle_20_Log', 'Spending_Second_20_Log', 'Spending_Gap_Lowest_Middle', 'Spending_Fourth_20_Sqrt']
✅ CSV saved: ../data/features/features.csv
✅ JSON (records) saved: ../data/features/features_records.json
✅ JSON (split) saved: ../data/features/features_split.json
✅ Pickle saved: ../data/features/features.pkl
✅ Parquet saved: ../data/features/features.parquet
✅ Feather saved: ../data/features/features.feather
✅ HDF5 saved: ../data/features/features.h5
✅ Metadata saved: ../data/features/features_metadata.json

✅ ALL FEATURES SAVED SUCCESSFULLY!

Saved files:
  - csv: ../data/features/features.csv
  - json_records: ../data/features/features_records.json
  - json_split: ../data/features/features_split.json
  - pickle: ../data/features/features.p

In [10]:

# VALIDATION AND QUALITY CHECKS


def validate_features(df, feature_list):
    
    
    print("\n" + "="*60)
    print("FEATURE VALIDATION REPORT")
    print("="*60)
    
    # Check data types
    print("\n🔹 Data Types Summary:")
    print(df[feature_list].dtypes.value_counts())
    
    # Check for missing values
    missing = df[feature_list].isnull().sum()
    if missing.sum() > 0:
        print("\n🔹 Missing Values Found:")
        print(missing[missing > 0])
    else:
        print("\n✅ No missing values in features")
    
    # Check for infinite values
    inf_check = df[feature_list].isin([np.inf, -np.inf]).sum()
    if inf_check.sum() > 0:
        print("\n🔹 Infinite Values Found:")
        print(inf_check[inf_check > 0])
    else:
        print("\n✅ No infinite values in features")
    
    # Feature statistics
    print("\n🔹 Feature Statistics:")
    stats = df[feature_list].describe()
    print(f"   Min values: {stats.loc['min'].min():.2f}")
    print(f"   Max values: {stats.loc['max'].max():.2f}")
    print(f"   Mean range: {stats.loc['mean'].min():.2f} to {stats.loc['mean'].max():.2f}")
    print(f"   Std range: {stats.loc['std'].min():.2f} to {stats.loc['std'].max():.2f}")
    
    # Check feature correlations (avoid duplicates)
    print("\n🔹 Checking for highly correlated features...")
    corr_matrix = df[feature_list].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr = [column for column in upper.columns if any(upper[column] > 0.95)]
    
    if high_corr:
        print(f"   ⚠️ {len(high_corr)} features have correlation > 0.95:")
        for col in high_corr[:5]:
            print(f"      - {col}")
    else:
        print("   ✅ No highly correlated features found")
    
    # Check feature variety
    print("\n🔹 Feature Categories:")
    feature_types = {
        'Temporal': [f for f in feature_list if 'Year' in f or 'Is_' in f or 'Since' in f],
        'Ratio': [f for f in feature_list if 'Ratio' in f],
        'Interaction': [f for f in feature_list if 'x_' in f or 'Interaction' in f],
        'Lagged': [f for f in feature_list if 'Lag' in f],
        'Rolling': [f for f in feature_list if 'Rolling' in f],
        'Transform': [f for f in feature_list if '_Log' in f or '_Sqrt' in f],
        'Domain': [f for f in feature_list if 'Exposure' in f or 'Pass_Through' in f],
        'Gap': [f for f in feature_list if 'Gap' in f],
        'Other': []
    }
    
    for feat_type, feats in feature_types.items():
        if feats:
            print(f"   {feat_type}: {len(feats)} features")
    
    # Check target correlation
    target_cols = ['Target_Spending_Gap', 'Target_Lowest_Energy_Spending',
                   'Target_Middle_Energy_Spending', 'Target_Energy_Poverty']
    
    print("\n🔹 Target Variables (for modeling):")
    for target in target_cols:
        if target in df.columns:
            print(f"   {target}:")
            print(f"      Range: {df[target].min():.2f} to {df[target].max():.2f}")
            print(f"      Mean: {df[target].mean():.2f}")
            
            # For classification targets
            if target in ['Target_Energy_Poverty', 'Target_Crisis_Flag']:
                class_dist = df[target].value_counts(normalize=True) * 100
                for cls, pct in class_dist.items():
                    print(f"      Class {cls}: {pct:.1f}%")
    
    print("\n" + "="*60)
    print("✅ VALIDATION COMPLETE")
    print("="*60)
    
    return True

# Run validation
validate_features(master_featured, top_features)

print("\n" + "="*60)
print("🏁 FEATURE ENGINEERING PIPELINE COMPLETE!")
print("="*60)
print(f"✅ Total features created: {len(master_featured.columns)}")
print(f"✅ Features selected: {len(top_features)}")
print(f"✅ Feature files saved in: ../data/features/")
print("\nAvailable formats:")
print("  - CSV (human-readable)")
print("  - JSON (web-friendly)")
print("  - Pickle (Python native)")
print("  - Parquet (efficient storage)")
print("  - Feather (fast I/O)")
print("  - HDF5 (hierarchical storage)")
print("\nNext steps:")
print("1. Load features from your preferred format")
print("2. Split into train/test sets")
print("3. Train ML models (Regression, Classification)")
print("4. Perform causal inference analysis")
print("5. Interpret results for policy recommendations")


FEATURE VALIDATION REPORT

🔹 Data Types Summary:
float64    29
int64       1
Name: count, dtype: int64

🔹 Missing Values Found:
Oil_Price_x_Highest_20                 48
Oil_Price_x_Middle_20                  48
Oil_Price_x_Lowest_20                  48
Spending_Middle_20_Rolling_Std_3yr     48
Spending_Lowest_20_Rolling_Std_3yr     48
Spending_Middle_20_YoY_Abs_Change      48
Spending_Highest_20_Rolling_Std_3yr    48
Spending_Lowest_20_YoY_Abs_Change      48
dtype: int64

✅ No infinite values in features

🔹 Feature Statistics:
   Min values: -1.49
   Max values: 15168.95
   Mean range: 0.19 to 7754.10
   Std range: 0.20 to 4642.59

🔹 Checking for highly correlated features...
   ⚠️ 18 features have correlation > 0.95:
      - Spending_Highest_20_Log
      - Spending_Fourth_20_Log
      - Spending_Highest_20_Rolling_Mean_3yr
      - Spending_Middle_20_Log
      - Spending_Second_20_Log

🔹 Feature Categories:
   Ratio: 1 features
   Interaction: 6 features
   Rolling: 6 features
   Tra